# 07 Results: structural intent: illusions named and measured

**Question.** Does the structural intent of the SDK name the illusion and answer the VLMBias
question correctly, on all 396 illusion images?

**Why.** VLMBias asks, for six classic illusions, whether two parts are equal. On the original
figure they are equal (answer Yes); on the edited figure one part is changed (answer No). A model
that recalls the classic figure answers Yes to both.

**Approach.** `scripts/run_structural.py` runs `Harness.answer` with the tools and intents of
`.env.paper` and a fixed classifier that chooses `structural`. Inside the intent, pixel rules name
the illusion from the image; that illusion's pipeline measures the two parts with the measurement
tools; `decide` answers Yes when the difference is inside the illusion's tolerance. The
tolerances are frozen: 1.5 times the largest difference measured on the originals. No language
model takes part. The images: 132 figures at three sizes (384, 768 and 1152 px), 198 original and
198 edited (`data/illusion-items.parquet`).

**Prediction.** The pixel rules name every image. The measurement answers every image correctly,
except Ponzo, where two originals at 384 px measure too long and set the tolerance above the
smallest edit (0.15).

In [1]:
import json
import sys
from pathlib import Path

import pandas as pd

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "scripts"))
import runs

RECORDED = runs.recorded_dir()
if RECORDED is None:
    print("INTENT_HARNESS_RECORDED_DIR is not set: the comparison with earlier results is skipped.")
else:
    print("INTENT_HARNESS_RECORDED_DIR is set: the comparison with earlier results runs.")

INTENT_HARNESS_RECORDED_DIR is set: the comparison with earlier results runs.


In [2]:
table = runs.read_result("02-structural")
named = int((table["named"] == table["gold_illusion"]).sum())
print("images:", len(table), "| named correctly:", named)

images: 396 | named correctly: 396


## 1. Accuracy by illusion and condition

In [3]:
rows = []
for illusion, part in table.groupby("gold_illusion"):
    originals = part[part["condition"] == "original"]
    edited = part[part["condition"] == "edited"]
    rows.append({
        "illusion": illusion,
        "originals correct": f"{int(originals['correct'].sum())} / {len(originals)}",
        "edited correct": f"{int(edited['correct'].sum())} / {len(edited)}",
        "tolerance": round(float(part["tolerance"].iloc[0]), 6),
    })
pd.DataFrame(rows)

,illusion,originals correct,edited correct,tolerance
0,ebbinghaus,36 / 36,36 / 36,0.000000
1,muller_lyer,36 / 36,36 / 36,0.184834
2,poggendorff,36 / 36,36 / 36,0.030414
3,ponzo,36 / 36,24 / 36,0.176471
4,vertical_horizontal,18 / 18,18 / 18,0.015544
5,zollner,36 / 36,36 / 36,0.000000


## 2. The images answered wrongly

In [4]:
wrong = table[~table["correct"]]
wrong.groupby(["gold_illusion", "edit_size", "pixel"]).size().rename("images").reset_index()

,gold_illusion,edit_size,pixel,images
0,ponzo,-0.15,384,2
1,ponzo,-0.15,768,2
2,ponzo,-0.15,1152,2
3,ponzo,0.15,384,2
4,ponzo,0.15,768,2
5,ponzo,0.15,1152,2


## 3. Comparison with the earlier results, item by item

Runs only when `INTENT_HARNESS_RECORDED_DIR` is set: the answer and the measured quantity of
each image against the earlier result files.

In [5]:
if RECORDED is not None:
    earlier = []
    for path in sorted(RECORDED.glob("1[4-9]-*-colour.parquet")):
        earlier.append(pd.read_parquet(path)[["item_id", "answer", "quantity"]])
    earlier = pd.concat(earlier)
    both = table.merge(earlier, on="item_id", suffixes=("", "_earlier"))
    print("images compared:", len(both))
    print("same answer:", int((both["answer"] == both["answer_earlier"]).sum()))
    print("largest difference in the quantity:", float((both["quantity"] - both["quantity_earlier"]).abs().max()))

images compared: 396
same answer: 396
largest difference in the quantity: 1.7763568394002505e-15


## Reading the result

The pixel rules name the illusion correctly on all 396 images.

Five of the six illusions are answered correctly on every image, original and edited. Ponzo is
correct on all 36 originals and on 24 of 36 edited images. The 12 wrong images are all the
smallest edit (0.15 bigger or 0.15 smaller), at all three image sizes. On these images the
measured difference is smaller than the frozen tolerance (0.176), so the pipeline answers Yes
("equal"). The answer is binary, so the harness cannot report a difference smaller than the
tolerance. This is a recorded limitation.

The comparison with the earlier results gives the same answer on 396 of 396 images. The largest
difference in a measured quantity is 1.8e-15, which is the floating-point limit.

## Conclusion

The structural intent of the SDK answers 384 of 396 illusion images correctly (0.970) and
reproduces the earlier results exactly. The only misses are the smallest Ponzo edits.